## Name: Mandar Sanju Bavdane

### CWID: 20041513                  Subject: FE_520 -- Assignment 2

In [264]:
# Install yfinance (needed for question 2 and 4)
!pip -q install yfinance

import yfinance as yf

# Question 1

In [265]:
news_block = """
     MSFT:
     Microsoft's quarterly earnings BEAT expectations, stock surges.
     ---
     AAPL:
     Apple announces major UPGRADE to its iPhone lineup, analysts are bullish.
     ---
     GOOG:
     Google faces potential DOWNGRADE amid regulatory concerns.
     ---
     META:
     Meta platforms MISS revenue targets, stock falls sharply.
     ---
     NVDA:
     NVIDIA's outlook strong, analysts reiterate BUY rating.
     """



1.1 Clean and split news_block

In [266]:
cleaned_snippets = [
    snippet.strip()
    for snippet in news_block.strip().split("---")
    if snippet.strip()
]

print("Cleaned news snippets:")
for snippet in cleaned_snippets:
    print(snippet)

Cleaned news snippets:
MSFT:
     Microsoft's quarterly earnings BEAT expectations, stock surges.
AAPL:
     Apple announces major UPGRADE to its iPhone lineup, analysts are bullish.
GOOG:
     Google faces potential DOWNGRADE amid regulatory concerns.
META:
     Meta platforms MISS revenue targets, stock falls sharply.
NVDA:
     NVIDIA's outlook strong, analysts reiterate BUY rating.


1.2 classify_sentiment(snippet)

In [267]:
def classify_sentiment(snippet):
    text = snippet.upper()

    if "BEAT" in text or "UPGRADE" in text or "BUY" in text:
        return "Bullish"
    elif "MISS" in text or "DOWNGRADE" in text:
        return "Bearish"
    else:
        return "Neutral"


# Demonstrate the function on one of the supplied snippets.
print(classify_sentiment("Earnings BEAT estimates"))
print(classify_sentiment("Revenue MISS"))
print(classify_sentiment("Stock flat today"))

Bullish
Bearish
Neutral


1.3 Ticker, Sentiment tuples

In [268]:
sentiment_results = []

for snippet in cleaned_snippets:
    ticker = snippet.split(":", 1)[0].strip()
    sentiment = classify_sentiment(snippet)
    sentiment_results.append((ticker, sentiment))

print("Sentiment results:", sentiment_results)

Sentiment results: [('MSFT', 'Bullish'), ('AAPL', 'Bullish'), ('GOOG', 'Bearish'), ('META', 'Bearish'), ('NVDA', 'Bullish')]


1.4 Overall Sentiment Score

In [269]:
overall_sentiment_score = 0

for ticker, sentiment in sentiment_results:
    if sentiment == "Bullish":
        overall_sentiment_score += 1
    elif sentiment == "Bearish":
        overall_sentiment_score -= 1

print("Overall sentiment score:", overall_sentiment_score)

Overall sentiment score: 1


1.5 Summary Report

In [270]:
bullish_count = sum(1 for ticker, sentiment in sentiment_results if sentiment == "Bullish")
bearish_count = sum(1 for ticker, sentiment in sentiment_results if sentiment == "Bearish")

sentiment_report = (
    f"News Sentiment Report\n"
    f"Total snippets analyzed: {len(cleaned_snippets)}\n"
    f"Bullish snippets: {bullish_count}\n"
    f"Bearish snippets: {bearish_count}\n"
    f"Overall sentiment score: {overall_sentiment_score:+d}"
)

print(sentiment_report)

News Sentiment Report
Total snippets analyzed: 5
Bullish snippets: 3
Bearish snippets: 2
Overall sentiment score: +1


# Question 2 - TSLA, first half of 2023

2.1 Download TSLA data and store the closing prices in list

In [271]:
tsla_data = yf.download(
    "TSLA",
    start="2023-01-01",
    end="2023-07-01", # to include 30 June's data too
    auto_adjust=False,
    progress=False
)

closing_prices = tsla_data["Close"].squeeze().dropna().tolist()

print("Number of closing prices:", len(closing_prices))
print("First five closing prices:", closing_prices[:5])
print("Last five closing prices:", closing_prices[-5:])

Number of closing prices: 124
First five closing prices: [108.0999984741211, 113.63999938964844, 110.33999633789062, 113.05999755859375, 119.7699966430664]
Last five closing prices: [241.0500030517578, 250.2100067138672, 256.239990234375, 257.5, 261.7699890136719]


2.2 Daily return (for loop) and standard deviation (without numpy/pandas)

In [272]:
daily_returns = []

for i in range(1, len(closing_prices)):
    daily_return = (closing_prices[i] - closing_prices[i - 1]) / closing_prices[i - 1]
    daily_returns.append(daily_return)


def calculate_std_dev(returns):
    if len(returns) == 0:
        return 0.0

    # First calculate the arithmetic mean of the returns
    mean_return = 0.0
    for value in returns:
        mean_return += value
    mean_return = mean_return / len(returns)

    # Calculate the sum of squared differences from the mean
    squared_difference_total = 0.0
    for value in returns:
        squared_difference_total += (value - mean_return) ** 2

    # Divide by n (population variance) and then sqrt
    variance = squared_difference_total / len(returns)
    standard_deviation = variance ** 0.5

    return standard_deviation


volatility = calculate_std_dev(daily_returns)

print("Number of daily returns:", len(daily_returns))
print("Daily return standard deviation:", f"{volatility:.4f}")

Number of daily returns: 123
Daily return standard deviation: 0.0355


2.3 Count high volatility days with a loop

In [273]:
high_volatility_days = 0
index = 0

while index < len(daily_returns):
    if abs(daily_returns[index]) > 0.04: # |return > 4%|
        high_volatility_days += 1
    index += 1

print("High Volatility days:", high_volatility_days)

High Volatility days: 33


2.4 Best and worst days using if statements

In [274]:
if len(daily_returns) > 0:
    best_day = daily_returns[0]
    worst_day = daily_returns[0]

    for daily_return in daily_returns:
        if daily_return > best_day:
            best_day = daily_return
        if daily_return < worst_day:
            worst_day = daily_return
else:
    best_day = 0.0
    worst_day = 0.0

print("Best day's return:", f"{best_day:.2%}")
print("Worst day's return:", f"{worst_day:.2%}")

Best day's return: 11.00%
Worst day's return: -9.75%


2.5 Summary Report

In [275]:
tsla_report = (
    f"TSLA Performance Report (First Half of 2023)\n\n"
    f"Trading days analyzed: {len(daily_returns)}\n"
    f"Volatility (standard deviation): {volatility:.4f}\n"
    f"High Volatility days: {high_volatility_days}\n"
    f"Best day's performance: {best_day:.2%}\n"
    f"Worst day's performance: {worst_day:.2%}"
)

print(tsla_report)

TSLA Performance Report (First Half of 2023)

Trading days analyzed: 123
Volatility (standard deviation): 0.0355
High Volatility days: 33
Best day's performance: 11.00%
Worst day's performance: -9.75%


# Question 3

In [276]:
trade_log = """
#Trade Log for 2025-10-03
 aapl, BUY, 100, 150.25
msft, BUY, 50, 300.50
 aapl, SELL, 25, 155.75


# A quick note on the next trade
GOOGL, buy, 10, 1350.00
msft, SELL, 50, 305.20
"""

3.1 parse_trade(log_line)

In [277]:
def parse_trade(log_line):
    cleaned_line = log_line.strip()

    if not cleaned_line or cleaned_line.startswith("#"):
        return None

    parts = [part.strip() for part in cleaned_line.split(",")]

    if len(parts) != 4:
        return None

    try:
        ticker = parts[0].upper()
        action = parts[1].upper()
        quantity = int(parts[2])
        price = float(parts[3])
    except ValueError:
        return None

    if not ticker or action not in ("BUY", "SELL"):
        return None

    return (ticker, action, quantity, price)


# Test the parser with one valid sample line.
print(parse_trade(" aapl, BUY, 100, 150.25 "))

('AAPL', 'BUY', 100, 150.25)


3.2 List of valid Trades

In [278]:
valid_trades = []

for line in trade_log.splitlines():
    parsed_trade = parse_trade(line)
    if parsed_trade is not None:
        valid_trades.append(parsed_trade)

print("Valid trades:")
for trade in valid_trades:
    print(trade)

Valid trades:
('AAPL', 'BUY', 100, 150.25)
('MSFT', 'BUY', 50, 300.5)
('AAPL', 'SELL', 25, 155.75)
('GOOGL', 'BUY', 10, 1350.0)
('MSFT', 'SELL', 50, 305.2)


3.3 Build the `portfolio`

In [279]:
portfolio = {}

for ticker, action, quantity, price in valid_trades:
    if ticker not in portfolio:
        portfolio[ticker] = 0

    if action == "BUY":
        portfolio[ticker] += quantity
    elif action == "SELL":
        portfolio[ticker] -= quantity

print("Final portfolio holdings:", portfolio)

Final portfolio holdings: {'AAPL': 75, 'MSFT': 0, 'GOOGL': 10}


3.4 `trade_counts`

In [280]:
trade_counts = {}

for ticker, action, quantity, price in valid_trades:
    if ticker not in trade_counts:
        trade_counts[ticker] = 0
    trade_counts[ticker] += 1

print("Trade counts:", trade_counts)

Trade counts: {'AAPL': 2, 'MSFT': 2, 'GOOGL': 1}


3.5 Summary Report

In [281]:
portfolio_report = "Portfolio Summary\n"

for ticker in portfolio:
    portfolio_report += (
        f"{ticker}: Final holdings = {portfolio[ticker]} shares, "
        f"Total trades = {trade_counts[ticker]}\n"
    )

print(portfolio_report)
print("Total trades:", len(valid_trades))

Portfolio Summary
AAPL: Final holdings = 75 shares, Total trades = 2
MSFT: Final holdings = 0 shares, Total trades = 2
GOOGL: Final holdings = 10 shares, Total trades = 1

Total trades: 5


# Question 4 - GOOGL Volume,
# November 2023

4.1 Download GOOGL data and store the volume in list

In [282]:
googl_data = yf.download(
    "GOOGL",
    start="2023-11-01",
    end="2023-12-01",
    auto_adjust=False,
    progress=False
)

volumes = googl_data["Volume"].squeeze().dropna().tolist()

print("Number of daily volume observations:", len(volumes))
print("First five daily volumes:", volumes[:5])

Number of daily volume observations: 21
First five daily volumes: [30082400, 27124600, 26393100, 19052700, 29757300]


4.2 Total volume with a for loop (without using `sum()`)

In [283]:
total_volume = 0

for daily_volume in volumes:
    total_volume += int(daily_volume)

print("Total trading volume:", total_volume)

Total trading volume: 535233300


4.3 Index of the first high volume day (>30 million) using `break`

In [284]:
first_high_idx = -1
for i in range(len(volumes)):
    if volumes[i] > 30_000_000:
        first_high_idx = i
        break

print("Index of first high-volume day:", first_high_idx)
if first_high_idx != -1:
    print("Date:", googl_data.index[first_high_idx].date(), "| Volume:", f"{volumes[first_high_idx]:,}")

Index of first high-volume day: 0
Date: 2023-11-01 | Volume: 30,082,400


4.4 Days needed to sell a 150 million share block using `while` loop

In [285]:
remaining_shares = 150000000
days_to_sell = 0
volume_index = 0

while remaining_shares >= 0 and volume_index < len(volumes):
    remaining_shares -= int(volumes[volume_index])
    days_to_sell += 1
    volume_index += 1

if remaining_shares >= 0:
    print("The available volume observations were insufficient to sell the full block.")
else:
    print("Days to sell the block:", days_to_sell)

Days to sell the block: 6


4.5 Summary Report

In [286]:
volume_report = f"""GOOGL Volume Summary (Nov 2023)
-----------------------------------
Total volume                 : {total_volume:,}
First high-volume day index  : {first_high_idx}
Days to sell 150M share block: {days_to_sell}"""
print(volume_report)

GOOGL Volume Summary (Nov 2023)
-----------------------------------
Total volume                 : 535,233,300
First high-volume day index  : 0
Days to sell 150M share block: 6


# Question 5

In [287]:
expenses = {
    "Rent": 1500,
    "Groceries": 450,
    "Utilities": 120,
    "Transport": 150
}

5.1 categorize_expense(amount)

In [288]:
def categorize_expense(amount):
    if amount > 300:
        return "High"
    elif 100 <= amount <= 300:
        return "Medium"
    else:
        return "Low"


print(categorize_expense(1500), categorize_expense(300), categorize_expense(100), categorize_expense(25))

High Medium Medium Low


5.2 Print each expense with its category

In [289]:
print("Current expenses:")
for item, amount in expenses.items():
    category = categorize_expense(amount)
    print(f" - {item}: ${amount} ({category})")

Current expenses:
 - Rent: $1500 (High)
 - Groceries: $450 (High)
 - Utilities: $120 (Medium)
 - Transport: $150 (Medium)


5.3 Add expenses interactively

Type entries like Coffee, 25 and so on

In [290]:
while True:
    user_entry = input("Add an expense (e.g., 'Coffee, 25') or type 'done': ").strip()

    if user_entry.lower() == "done":
        break

    if "," not in user_entry:
        print("Error: Please enter the expense name and amount separated by a comma.")
        continue

    item, amount_text = user_entry.split(",", 1)
    item = item.strip()
    amount_text = amount_text.strip()

    try:
        amount = int(amount_text)
        if not item:
            print("Error: Please enter an expense name.")
            continue
        expenses[item] = amount
        print(f"Added {item}: ${amount}")
    except ValueError:
        print("Error: The expense amount must be a whole number.")

Add an expense (e.g., 'Coffee, 25') or type 'done': Coffee, 25
Added Coffee: $25
Add an expense (e.g., 'Coffee, 25') or type 'done': Stationary, 100
Added Stationary: $100
Add an expense (e.g., 'Coffee, 25') or type 'done': Beverages, 30
Added Beverages: $30
Add an expense (e.g., 'Coffee, 25') or type 'done': Done


5.4 calculate_total(expense_dict)

In [291]:
def calculate_total(expense_dict):
    total = 0

    for amount in expense_dict.values():
        total += amount

    return total


print("Current monthly total:", calculate_total(expenses))

Current monthly total: 2375


5.5 Final Summary

In [292]:
total_cost = calculate_total(expenses)

print("Updated expenses:")
for item, amount in expenses.items():
    print(f" - {item}: ${amount:g} ({categorize_expense(amount)})")

print(f"""
Monthly Expense Summary
-----------------------
Total expense items : {len(expenses)}
Total monthly cost  : ${total_cost:,.2f}""")

Updated expenses:
 - Rent: $1500 (High)
 - Groceries: $450 (High)
 - Utilities: $120 (Medium)
 - Transport: $150 (Medium)
 - Coffee: $25 (Low)
 - Stationary: $100 (Medium)
 - Beverages: $30 (Low)

Monthly Expense Summary
-----------------------
Total expense items : 7
Total monthly cost  : $2,375.00
